### Authenticate
Get the client id and secret from env vars for prod.
In dev we can create a new client each time

In [1]:
import { registerSystem } from '../v1-to-v2-data-migration/helpers/gqlHandlers.ts'
import { authenticate, getTokenForSystemClient } from '../v1-to-v2-data-migration/helpers/authentication.ts'
import { CLIENT_ID, CLIENT_SECRET } from '../v1-to-v2-data-migration/helpers/vars.ts'
import { ADMIN_USERNAME, ADMIN_PASSWORD } from '../v1-to-v2-data-migration/helpers/vars.ts'


let clientId: string | null = null
let clientSecret: string | null = null

if (CLIENT_ID && CLIENT_SECRET) {
  clientId = CLIENT_ID
  clientSecret = CLIENT_SECRET
} else {
  // For dev mode
  const adminToken = await authenticate(ADMIN_USERNAME, ADMIN_PASSWORD)
  const systemRegistration = await registerSystem(adminToken)

  clientId = systemRegistration.data.registerSystem.system.clientId
  clientSecret = systemRegistration.data.registerSystem.clientSecret
}

const sysToken = await getTokenForSystemClient(clientId, clientSecret)
sysToken

"eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJzY29wZSI6WyJyZWNvcmQuaW1wb3J0IiwicmVjb3JkLmV4cG9ydCIsInJlY29yZHNlYXJjaCIsInVzZXIuZGF0YS1zZWVkaW5nIiwicmVjb3JkLnJlaW5kZXgiLCJkZW1vIl0sInVzZXJUeXBlIjoic3lzdGVtIiwiaWF0IjoxNzY5Njk1NDYxLCJleHAiOjE3NzAzMDAyNjEsImF1ZCI6WyJvcGVuY3J2czphdXRoLXVzZXIiLCJvcGVuY3J2czp1c2VyLW1nbnQtdXNlciIsIm9wZW5jcnZzOmhlYXJ0aC11c2VyIiwib3BlbmNydnM6Z2F0ZXdheS11c2VyIiwib3BlbmNydnM6bm90aWZpY2F0aW9uLXVzZXIiLCJvcGVuY3J2czp3b3JrZmxvdy11c2VyIiwib3BlbmNydnM6c2VhcmNoLXVzZXIiLCJvcGVuY3J2czptZXRyaWNzLXVzZXIiLCJvcGVuY3J2czpjb3VudHJ5Y29uZmlnLXVzZXIiLCJvcGVuY3J2czp3ZWJob29rcy11c2VyIiwib3BlbmNydnM6Y29uZmlnLXVzZXIiLCJvcGVuY3J2czpkb2N1bWVudHMtdXNlciJdLCJpc3MiOiJvcGVuY3J2czphdXRoLXNlcnZpY2UiLCJzdWIiOiI2OTdiNjhlNTRjZjI4NTVlMTM2N2VhNmQifQ.a5XW9Ek0K5F5OjYGYOGZsc-Exc9Zl14WhGIZRPudxrL4IrguuxLO6EEfXSPKFK0BzAr4MTTdeJj8u9mCAMSqWRdhMZqmKAPex9C3_2jKWj7wEe7S7m2nk_EM1b5jPTI-JnSkZgE-TzoiSUmxNsax7HZQ36RAY8ywsNDkrss9yvRK-GXr570DDJ_8DIdtj_Gs-BACoX3SbHL1SLLkWmEy_f2Dev1C_y3yJGaI6O6mIuc4-DHrNR0SH7nH6-tEzh4FQxO

### Fetch data from CSVs

In [2]:
import { csvToJson } from './helpers/csvHelpers.ts'

const pathToBirthCsv = './sourceData/Birth_Register.csv'
const pathToDeathCsv = './sourceData/Death_Register.csv'
const pathToMarriageCsv = './sourceData/Marriage_Register.csv'
const pathToAdoptionCsv = './sourceData/Adoption_Register.csv'
const pathToDeedpollCsv = './sourceData/Deedpoll.csv'

const csvData = {
  birth: await csvToJson(pathToBirthCsv),
  death: await csvToJson(pathToDeathCsv),
  marriage: await csvToJson(pathToMarriageCsv),
  adoption: await csvToJson(pathToAdoptionCsv),
  deedpoll: await csvToJson(pathToDeedpollCsv),
}


In [ ]:
import { GATEWAY } from "../v1-to-v2-data-migration/helpers/routes.ts";
import { locationsMap} from './lookupMappings/locations.ts'

export const getLocations = async (token: string) => {
  const response = await fetch(`${GATEWAY}/location?type=ADMIN_STRUCTURE&_count=0&status=active`, {
    method: 'GET',
    headers: {
      'Content-Type': 'application/json',
      Authorization: `Bearer ${token}`,
    },
  })
  if (!response.ok) {
    throw new Error(`Sync Locations failed: ${response.statusText}`)
  }
  return response
}

const locationsRes = await getLocations(sysToken)
const fhirLocations = await locationsRes.json()
const locationCodes = fhirLocations.entry.map((loc: any) => ({
  id: loc.resource.id,
  name: loc.resource.name,
  code: loc.resource.description
}))

//locationCodes

locationsMap.map((loc: any) => ({
  ...loc,
  id: locationCodes.find((l: any) => l.code === loc.map)?.id || null,
}))



[
  {
    name: '"MV AKATERE" EN ROUTE TO PUKAPUKA',
    map: "COK-011",
    id: "a659ade6-9a9a-427e-8067-dc97bab9fd1a"
  },
  { name: '"TAVEUNI" AT SEA', map: null, id: null },
  { name: "''TIARE TAPORO'' AT SEA", map: null, id: null },
  {
    name: "(1) MAKATEA ISLAND (2) FRENCH OCEANIA",
    map: null,
    id: null
  },
  { name: "(AT WAR)", map: null, id: null },
  {
    name: ",ITIARO",
    map: "COK-005",
    id: "9a8afa24-5bfb-4157-b658-0ee77faf1aa9"
  },
  {
    name: ",MANGAIA",
    map: "COK-006",
    id: "ee8b3512-f792-4ff1-94ac-00af70794919"
  },
  {
    name: "-",
    map: "COK-001-001-001",
    id: "9ba4c2b2-0258-45c2-98f2-3d0b882790f8"
  },
  {
    name: "---------------",
    map: "COK-001-001-001",
    id: "9ba4c2b2-0258-45c2-98f2-3d0b882790f8"
  },
  {
    name: ".ARORANGI,RAROTONGA",
    map: "COK-001-005",
    id: "d0ceda0f-ffbc-4c10-b873-263faef2b483"
  },
  { name: "0", map: null, id: null },
  {
    name: "00000000000000000000000000000000000000000000000000",
   

### Get all potential resolvers
Use only resolvers for used event fields to avoid nulls

In [5]:
import { birthResolver } from './mappings/birthResolver.ts'
import { transform } from './helpers/transform.ts'
import { bulkImport } from '../v1-to-v2-data-migration/helpers/gqlHandlers.ts'
import {
  batch,
  getIndexErrors,
} from '../v1-to-v2-data-migration/helpers/utils.ts'

function nonNullObjectKeys(obj: Record<string, any>) {
  return Object.fromEntries(
    Object.entries(obj).filter(
      ([_, value]) => value !== null && value !== undefined && value !== '',
    ),
  )
}

const events = []

csvData.birth.forEach((birth) => {
  const declaration: typeof birthResolver = {}
  Object.entries(birthResolver).forEach(([eventField, dataField]) => {
    if (dataField) {
      const data = dataField(birth, csvData, locationMap)
      declaration[eventField] = data
    }
  })

  const user = 'f686c526-c6b5-41ed-b3cc-43b104fa5c03'
  const location = '08260e22-bb67-4702-8760-86ec125e1079'
  const role = 'REGISTRAR'
  const event = transform(
    nonNullObjectKeys(declaration),
    'birth',
    new Date(),
    user,
    role,
    location,
    birth.BIRTH_REF,
  )
  events.push(event)
})
const batches = batch(events, 1000)
//console.log(events.slice(0, 1))

for (const batch of [batches[0]]) {
  const res = await bulkImport(batch, sysToken)
  const errors = getIndexErrors(res)
  if (errors) {
    console.error('Errors during bulk import', errors)
  }  
}


### Migrate births

### Migrate Deaths

In [6]:
import { reindex } from "../v1-to-v2-data-migration/helpers/gqlHandlers.ts";

const reindexResponse = await reindex(sysToken);
reindexResponse


"OK"

### Output results


In [7]:
console.log('🍞 Declarations succesfully migrated:')


🍞 Declarations succesfully migrated:
